# q2a

Two pieces:

1. Replicate "receive $S_T$ at $T$" (the long share leg).
2. Replicate "pay $K$ at $T$" (the short cash leg).

Leg 1: receive $S_T$ at $T$. 

At time $t$, hold $e^{-q(T-t)}$ shares of $S$. Because dividends are reinvested, by time $T$ we'll hold

$$e^{-q(T-t)} \cdot e^{q(T-t)} \;=\; 1 \text{ share}$$

worth $S_T$ dollars. The time-$t$ cost is $e^{-q(T-t)} S_t$ dollars.

Leg 2: pay $K$ at $T$.

 Short $K e^{-r(T-t)}$ dollars in the bank account. At $T$, that grows to $K$ dollars, which exactly funds the payment. The time-$t$ cost is $-K e^{-r(T-t)}$ (we *receive* this much now).

Replicating portfolio at time $t$:

$$f_t \;=\; e^{-q(T-t)}\,S_t \;-\; K e^{-r(T-t)}$$

so,

$$0 \;=\; e^{-q(T-t)} S_t - F_t\, e^{-r(T-t)}$$
$$F_t = S_t\, e^{(r-q)(T-t)}$$



# q2b

Consider bundle:

$$\text{1 share}$$
$$\text{1 share} + D e^{-r T_0} \text{units of bank account}$$

so,

$$b_t \;=\; \begin{cases}S_t, & t < T_0,\\[2pt]S_t + D\, e^{-r T_0} \cdot e^{r t} = S_t + D\, e^{r(t - T_0)}, & t \ge T_0.\end{cases}$$

### case $t \ge T_0$:

Shares of S: 1

Bank account $-K e^{-r(T-t)}$

Value: $f_t = S_t - K e^{-r(T-t)}$.

Setting $f_t = 0$ gives $\;F_t = S_t e^{r(T-t)}\;$ for $t \ge T_0$.

### case $t < T_0$:

1 share, worth $S_T$ dollars

$D$ dollars received at $T_0$, which by time $T$ has grown to $D e^{r(T - T_0)}$ in bank account.

so,

Shares of S: 1

Bank account $-D e^{-r(T_0 - t)} - K e^{-r(T-t)}$

Value: $f_t = S_t - D e^{-r(T_0 - t)} - K e^{-r(T-t)}$.

Setting $f_t = 0$:

$$F_t\, e^{-r(T-t)} \;=\; S_t - D e^{-r(T_0 - t)}$$

$$\;F_t \;=\; S_t e^{r(T-t)} - D e^{r(T - T_0)}$$

when combined:

$$F_t \;=\; \begin{cases}S_t e^{r(T-t)} & \text{if } t \ge T_0,\\[2pt]S_t e^{r(T-t)} - D e^{r(T-T_0)} & \text{if } t < T_0.\end{cases}$$



In [10]:
import numpy as np
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [11]:
class CEV:

    # Allow X0 to be initially unspecified.  Because we may want to use X to model a forward price X0 that will be calculated from a spot price.
    def __init__(self,volcoeff,alpha,rGrow,r,X0=None):
        self.volcoeff = volcoeff
        self.alpha = alpha
        self.rGrow = rGrow
        self.r = r
        self.X0 = X0


In [12]:
class discreteDividendModel:

    def __init__(self,S0,T,forwardDynamics,discreteDivDate,discreteDivAmt):
        self.discreteDivDate = discreteDivDate
        self.discreteDivAmt = discreteDivAmt
        self.S0 = S0
        self.T = T
        self.forwardDynamics = forwardDynamics
        self.forwardDynamics.X0 = self.convertSpotToForward(S0,0)

    def getX0(self):
        return self.forwardDynamics.X0

    def convertSpotToForward(self,S,t,tBumpToDetectDivdate=1e-12):
        r = self.forwardDynamics.r
        T = self.T
        T0 = self.discreteDivDate
        D = self.discreteDivAmt
        # time t=T0 has two "sides": immediately before and immediately after the transition of S, from including to not including the dividend.
        # The sign of tBumpToDetectDivdate indicates which side we are on at time t=T0.
        if t + tBumpToDetectDivdate > T0:
            return S*np.exp(r*(T-t))
        else:
            return S*np.exp(r*(T-t))-D*np.exp(r*(T-T0))

    def convertForwardToSpot(self,X,t,tBumpToDetectDivdate=1e-12):
        r = self.forwardDynamics.r
        T = self.T
        T0 = self.discreteDivDate
        D = self.discreteDivAmt
        # time t=T0 has two "sides": immediately before and immediately after the transition of S, from including to not including the dividend.
        # The sign of tBumpToDetectDivdate indicates which side we are on at time t=T0.
        if t + tBumpToDetectDivdate > T0:
            return X * np.exp(-r * (T - t))
        else:
            return X * np.exp(-r * (T - t)) + D * np.exp(-r * (T0 - t))

        # These are the only two lines of code that need to be completed.
        # Hint... do the "opposite" of what convertSpotToForward does

In [13]:
hw6ForwardDynamics = CEV(volcoeff=3, alpha=-0.5, rGrow=0, r=0.05)
hw6discreteDividendModel = discreteDividendModel(S0=100,T=0.25,forwardDynamics=hw6ForwardDynamics,discreteDivDate=0.15,discreteDivAmt=2)

In [14]:
class Call:

    def __init__(self,T,K,American=False):
        self.T = T
        self.K = K
        self.American = American

In [15]:
hw6contract=Call(T=0.25,K=100,American=True)

In [16]:
class FD_CrankNicolson_Engine:

    def __init__(self,XMax,XMin,deltaX,deltat):
        self.XMax=XMax
        self.XMin=XMin
        self.deltaX=deltaX
        self.deltat=deltat

    def TicksAndMatricesCEV(self,T,dynamics):

        alpha, r, rGrow, volcoeff = dynamics.alpha, dynamics.r, dynamics.rGrow, dynamics.volcoeff

        N=round(T/self.deltat)
        if abs(N-T/self.deltat)>1e-12:
            raise ValueError('Bad time step')
        numX=round((self.XMax-self.XMin)/self.deltaX)+1
        if abs(numX-(self.XMax-self.XMin)/self.deltaX-1)>1e-12:
            raise ValueError('Bad space step')
        X=np.linspace(self.XMax,self.XMin,numX)    #The FIRST indices in this array are for HIGH levels of X
        tTicks = np.arange(N-1,-1,-1)*self.deltat

        ratio1 = self.deltat/self.deltaX
        ratio2 = self.deltat/self.deltaX**2
        f = (1 / 2) * (volcoeff ** 2) * (X ** (2 * (1 + alpha)))
        g = rGrow * X
        h = -r * np.ones(np.size(X))          ### Scalar also acceptable here
        F = 0.5*ratio2*f + 0.25*ratio1*g
        G =     ratio2*f - 0.50*self.deltat*h
        H = 0.5*ratio2*f - 0.25*ratio1*g

        #Right-hand-side matrix
        RHSmatrix = diags([H[:-1], 1-G, F[1:]], [1,0,-1], shape=(numX,numX), format="csr")

        #Left-hand-side matrix
        LHSmatrix = diags([-H[:-1], 1+G, -F[1:]], [1,0,-1], shape=(numX,numX), format="csr")
        # diags creates SPARSE matrices

        return(X, tTicks, LHSmatrix, RHSmatrix, H[-1], F[0])

    def price_call_CEV(self,contract,discreteDividendModel):

        # returns array of all initial X levels,
        # and the corresponding array of call prices

        if discreteDividendModel.T < contract.T:
            raise ValueError('The forward price cannot be for a delivery date earlier than the expiry of the option')

        dynamics = discreteDividendModel.forwardDynamics

        X, tTicks, LHSmatrix, RHSmatrix, bottomH, topF = self.TicksAndMatricesCEV(contract.T,dynamics)
        # The X array contains the _interior_ levels of the grid,
        # from the smallest XMin to the largest XMax
        # The boundary conditions are imposed one level _beyond_,
        # e.g. at X_lowboundary=XMin-deltaX, not at XMin.
        # To relate to lecture notation, X_lowboundary is X_{-J}
        # whereas XMin is X_{-J+1}

        callprice=np.maximum(discreteDividendModel.convertForwardToSpot(X,contract.T)-contract.K,0)
        X_highboundary=self.XMax+self.deltaX

        for t in tTicks:

            rhs = RHSmatrix * callprice

            #Now let's add the boundary condition vectors.
            #They are nonzero only in the first component:
            rhs[0]=rhs[0]+2*topF*(discreteDividendModel.convertForwardToSpot(X_highboundary,t)-contract.K*np.exp(-dynamics.r*(contract.T-t)))
            #Strictly speaking, this aggregation of boundary conditions at time t and t+dt should have a t term plus a t+dt term, rather than 2*(the t term)
            #But the difference is negligible

            callprice = spsolve(LHSmatrix, rhs)  #You code this.  Hint...
            # numpy.linalg.solve, which expects arrays as inputs,
            # is fine for small matrix equations, and for matrix equations without special structure.
            # But for large matrix equations in which the matrix has special structure,
            # we may want a more intelligent solver that can run faster
            # by taking advantage of the special structure of the matrix.
            # Specifically, in this case, let's try to use a solver that recognizes the SPARSE MATRIX structure.
            # Try spsolve, imported from scipy.sparse.linalg

            if contract.American:
                if abs(t-discreteDividendModel.discreteDivDate)<self.deltat/2:  #if t is the dividend date
                    #then check whether it's better to exercise immediately before the stock price drops by the dividend amount
                    callprice = np.maximum(callprice, discreteDividendModel.convertForwardToSpot(X,t,-self.deltat/2)-contract.K)

        return(discreteDividendModel.convertForwardToSpot(X,t), X, callprice)

In [17]:
X0 = hw6discreteDividendModel.getX0()
deltaX = 0.1
hw6FD = FD_CrankNicolson_Engine(XMax=X0+deltaX*1000,XMin=X0-deltaX*500,deltaX=deltaX,deltat=0.0005)

In [18]:
(S0_all, X0_all, callprice) = hw6FD.price_call_CEV(hw6contract,hw6discreteDividendModel)

In [19]:
# price_call_CEV gives us option prices for ALL X0 from XMin to XMax
# But let's display only a few rows

import pandas as pd

df = pd.DataFrame({
    'Spot Price (S0)': S0_all,
    'Forward Price (X0)': X0_all,
    'Call Price': callprice
})

df_display = df[(df['Spot Price (S0)'] > hw6discreteDividendModel.S0 - hw6FD.deltaX * 1.5)
              & (df['Spot Price (S0)'] < hw6discreteDividendModel.S0 + hw6FD.deltaX * 1.5)]
df_display

,Spot Price (S0),Forward Price (X0),Call Price
999,100.098758,99.34782,5.827586
1000,100.000000,99.24782,5.775510
1001,99.901242,99.14782,5.723719
